## 第12章 二进制和序列化

### 1.二进制

- 在计算机中， 二进制、八进制、十进制、十六进制只是整数的不同表示方式，**绑定的实际值相同**。
    - 二进制：用0-1表示数据，`0b`前缀标记。`bin()`函数可以将整数转换为二进制表示。
    - 八进制：用0-7表示数据，`0o`前缀标记。`oct()`函数可以将整数转换为八进制表示。
    - 十六进制：用0-9和A-F(10-15)表示数字，`0x`前缀标记。`hex()`函数可以将整数转换为十六进制表示。

        <p><img src="img/003.jpg" width="650"></p>

- 补码：在大多数计算机中，负数表示为正数的二进制**补码**。计算补码的方法是：
    - 写出正数的二进制：42 → `0b00101010`。
    - 逐位取反：→ `0b11010101`。
    - 加1：→ `0b11010110`(-42)。
- Python3的`int`是**无限精度**的，负数的补码理论上需要无限个前导1，因此Python用特殊方式显示负数二进制：对正数的二进制表示前置一个负号来显示负的二进制数字。

In [ ]:
print(bin(42))      # 0b101010
print(bin(-42))     # -0b101010，Python用正数前再加一个负号来表示二进制负数

- 字节序：在计算机中，大多数数据由多个字节组成，而字节序是指字节在内存中的存储顺序。分为两种顺序(可以通过`sys.byteorder`查看当前字节序)：
    - 大端序：高位字节存在最低内存地址，**与人类书写顺序一致**。
    - 小端序：低位字节存在最低内存地址，**Intel/AMD 处理器默认使用**。

- 位运算：
    - `&`：按位与操作，如果左、右操作数中对应位均为1，则结果还是1，否则为0。
    - `|`：按位或操作，只要左、右操作数的对应位中有一位为1，则结果为1。
    - `^`：按位异或操作，如果某个位在任一操作数中为1，在另一操作数中为0，则结果为1。
    - `~`：按位取反操作，翻转给定操作数中的每一位。
        - 在Python中，由于整数是无限的，新值将有无限个前导1，由于无穷多的1难以输出，因此以负二进制表示法来显示结果：**在前面放置一个负号，并减去1以绕过二进制补码。**
    - `<<`：左移操作，将操作数的二进制表示向左移动指定的位数。
    - `>>`：右移操作，将操作数的二进制表示向右移动指定的位数。
        - Python用**算术移位**，右移保留符号位(正数补0，负数补1)。
    
        <p><img src="img/004.jpg" width="650"></p>

In [ ]:
print(bin(0b1101 & 0b1010))     # 0b1000
print(bin(0b1101 | 0b1010))     # 0b1111
print(bin(0b1101 ^ 0b1010))     # 0b0111
print(bin(~0b1101))             # -0b1110
print(bin(0b1100 << 4))         # '0b11000000'（左移补0）
print(bin(0b1100 >> 4))         # '0b0'（正数右移补0）
print(bin(-0b1100 >> 4))        # '-0b1'（负数右移补1，等效无限前导1）

- 字节字面量：Python中表示二进制的另一种方法，以`b`为作为前缀，每个元素为ASCII字符或十六进制转义`\xhh`。如`b'hello'`或`b'\x48\x65\x6c\x6c\x6f'`。
    - 字节字面量只能包含ASCII字符(0x00-0xFF)。
    - 字节字面量不能通过f-字符串进行格式化。
    - `\x`后必须跟两位十六进制，大小写不敏感。

### 2.类字节对象

- 类字节对象：用于表示二进制数据的类，包括不可变的`bytes`、可变的`bytearray`，该类对象不能进行位操作。创建方法：
    - `bytes(n)`：创建n个空字节的bytes对象。注意：`bytes(0b110)`创建的是6个空字节，**不是**值为0b110的bytes！
    - `bytes((0b110,))`：单元素元组包裹整数。
    - `bytes([0b110])`：列表包裹整数。
    - `b'\x06'`：bytes字面量直接赋值。
    - `bytearray(b'\x06')`：转为bytearray对象。
    - `bytes('⊗', encoding='utf-8')`：字符串按指定编码转bytes对象。


In [ ]:
print(bytes(0b110))                     # b'\x00\x00\x00\x00\x00\x00'，创建6个空字节的bytes对象
print(bytes((0b110,)))                  # b'\x06'
print(bytes('⊗', encoding='utf-8'))    # b'\xe2\x96\x94'

- 整数与字节对象的转换：
    - `int.to_bytes(n, byteorder='big')`：整数转为n字节的bytes对象。
    - `int.from_bytes(bytes, byteorder='big')`：将bytes对象转换为整数。

In [ ]:
import sys

bits = (42).to_bytes(4, byteorder=sys.byteorder)     # 按照当前系统字节序，转为4字节的bytes对象
print(bits.hex())                                    # 2a000000

bits = (-42).to_bytes(4, byteorder=sys.byteorder, signed=True)    # 如果为负数，必须指定signed=True
print(bits.hex())                                                 # d6ffffff，二进制补码表示

num = int.from_bytes(bits, byteorder=sys.byteorder, signed=True)  # 从bytes对象转换为整数
print(num)

### 3.struct模块

- struct模块最初用于Python值与C结构体之间的数据交换，现已成处理**打包二进制数据**的常用工具，在打包或解包时必须指定格式字符串，包括字节序、对齐行为、数据类型等。
    - `struct.pack(format, ...args)`：将Python值打包为二进制数据，返回bytes对象。
    - `struct.unpack(format, bytes)`：将二进制数据解包为Python值，返回元组。

        <p><img src="img/005.jpg" width="650"></p>

        <p><img src="img/006.jpg" width="650"></p>

In [ ]:
import struct

# 打包为二进制数据
bits = struct.pack('>2i?', 4, 2, True)      # 将4,2,True按照“大端序，2个int，1个bool”打包为二进制数据
print(bits)                                 # b'\x00\x00\x00\x04\x00\x00\x00\x02\x01'

bits = struct.pack('>i3xi', -4, -2)         # 将-4,-2按照“大端序，1个int，3个空字节填充，1个int”打包为二进制数据
print(bits)                                 # b'\xff\xff\xff\xfc\x00\x00\x00\xff\xff\xff\xfe'

bits = struct.pack('>4s', b"Hi!")           # C字符串：以 \x00 结尾标记结束
print(bits)                                 # b'Hi!\x00'

bits = struct.pack('>4p', b"Hi!")           # Pascal字符串：首字节存储长度，无需空终止符，最大255字节
print(bits)                                 # b'\x03Hi!'

# 解包二进制数据
bits = struct.pack('i', -360)
nums, = struct.unpack('i', bits)            # 必须尾随逗号！因为unpack返回元组
print(nums)                                 # -360

bits = struct.pack('>i3xi', -4, -2)
first, second = struct.unpack('>i3xi', bits)
print(first, second)                          # -4 -2

### 4.二进制文件

- 二进制流模式：流必须以**二进制模式**打开，返回 `BufferedReader`/`BufferedWriter`/`BufferedRandom` 对象。
- 二进制流的`seek()`比文本流更强大，`whence`有三个值：0从**开头**开始(偏移量必须是非负数)，1从**当前位置**开始，2从**末尾**开始(应使用负偏移量)。
- `BufferedRWPair` 接受两个流对象（一个读，一个写），主要用途是与套接字或双向管道通信。

In [ ]:
import struct
from dataclasses import dataclass

# 书类
@dataclass
class Book:
    title: str
    author: str
    pages: int = 0
    pages_read: int = 0

    packer = struct.Struct('>64sx64sx2h')  # 大端序，64字节标题，1个空字节，64字节作者，1个空字节，2个sho

    # 序列化: 将对象转换为字节序列
    def serialize(self):
        return self.packer.pack(
            self.title.encode(),
            self.author.encode(),
            self.pages,
            self.pages_read
        )

    # 反序列化: 将字节序列转换为对象
    @classmethod
    def deserialize(cls, bits):
        title, author, pages, pages_read = cls.packer.unpack(bits)
        return cls(title.decode(), author.decode(), pages, pages_read)

# 书架类
class Bookshelf:
    fileinfo = struct.Struct('>h')  # 大端序，1个short
    version = 1  # 版本号

    def __init__(self, *books):
        self.shelf = [*books]

    def __iter__(self):
        return iter(self.shelf)

    def add_books(self, *books):
        self.shelf.extend(books)

    # 将书架对象序列化为字节序列并写入流
    def to_stream(self, stream):
        stream.write(self.fileinfo.pack(self.version))
        for book in self.shelf:
            stream.write(book.serialize())

    # 从流中读取字节序列并反序列化为书架对象
    @classmethod
    def from_stream(cls, stream):
        # 读取版本号
        size = cls.fileinfo.size
        version, = cls.fileinfo.unpack(stream.read(size))
        if version != 1:
            raise ValueError(f'版本号错误，期望{cls.version}，实际{version}')

        # 读取书架中的所有书
        size = Book.packer.size
        shelf = Bookshelf()
        while bits := stream.read(size):
            shelf.add_books(Book.deserialize(bits))
        return shelf

def write_demo_file():
    bookshelf = Bookshelf(
        Book("Automate the Boring Stuff with Python", "Al Sweigart", 592, 592),
        Book("Doing Math with Python", "Amit Saha", 264, 100),
        Book("Black Hat Python", "Justin Seitz", 192, 0),
        Book("Serious Python", "Julien Danjou", 240, 200),
        Book("Real-World Python", "Lee Vaughan", 370, 370),
    )
    with open('res/bookshelf.bin', 'wb') as f:
        bookshelf.to_stream(f)

def read_demo_file():
    with open('res/bookshelf.bin', 'rb') as f:
        bookshelf = Bookshelf.from_stream(f)
        for book in bookshelf:
            print(book.title)

if __name__ == '__main__':
    write_demo_file()
    read_demo_file()

### 5.本章小结

- **核心知识脉络**

```text
二进制与序列化
│
├── 1. 二进制基础
│   ├── 二进制字面量：0b 前缀
│   ├── 位值表（128→1）
│   └── 字节 = 8位（通常）
├── 2. 十六进制
│   ├── 0x 前缀，A-F 表示 10-15
│   ├── 转换公式：位值 = 16^n × 数字
│   └── bin() / hex() / oct() 转换函数
├── 3. 八进制
│   ├── 0o 前缀，0-7 表示
│   └── 不同进制位值对比表
├── 4. 整数与多进制
│   └── 不同进制只是同一整数的不同表示方式
├── 5. 二进制补码
│   ├── 正数取反+1 = 负数补码
│   ├── Python 无限精度 int 的特殊显示：-0b101010
│   └── 位掩码查看8位补码：-42 & 0b11111111
├── 6. 字节序 ★★★
│   ├── 大端序：高位存低地址（人类书写顺序）
│   ├── 小端序：低位存低地址（Intel/AMD 默认）
│   └── 网络传输统一用大端序
├── 7. 位运算
│   ├── 运算符：& | ^ ~ << >>
│   ├── 算术移位：右移保留符号位
│   └── Python int 无限精度的影响
├── 8. Bytes 字面量
│   ├── b 前缀，ASCII / \xhh 十六进制转义
│   └── 原始 bytes：rb / br 前缀
├── 9. Bytes-Like 对象
│   ├── bytes（不可变）/ bytearray（可变）
│   ├── 不支持位运算（字节序未知）
│   └── 6种创建方式（⚠️ bytes(n) 是陷阱！）
├── 10. int ↔ bytes 互转
│   ├── int.to_bytes(size, byteorder, signed=)
│   ├── int.from_bytes(bytes, byteorder, signed=)
│   └── ⚠️ 必须手动指定字节序和 signed
├── 11. struct 模块 ★★★
│   ├── 字节序标志：@ = < > !
│   ├── 格式字符：i/f/s/p/x 等
│   ├── pack() / unpack() / Struct 对象复用
│   └── ⚠️ 解包格式字符串必须与打包时完全一致
├── 12. Bytes-Like 位运算变通
│   ├── 整数中转法：int.from_bytes() → 位运算 → to_bytes()
│   └── 迭代法：zip() 逐字节运算（内存友好）
├── 13. memoryview
│   ├── 切片不创建副本，原地访问原始内存
│   └── with 语句自动释放（推荐）
├── 14. 二进制文件 I/O
│   ├── 模式：wb / rb / ab
│   ├── seek(whence=0/1/2)
│   └── BufferedRWPair：双流读写
├── 15. 序列化技术
│   ├── pickle ⚠️（不安全，仅限内部使用）
│   ├── shelve ⚠️（基于 pickle，字典式持久化）
│   ├── plistlib（Property List）
│   ├── MessagePack / Protocol Buffers
│   └── json（跨语言首选，安全）
└── 16. 综合项目：Bookshelf
    ├── Book 类：struct 打包/解包 + encode/decode
    └── Bookshelf 类：流式读写 + 版本号校验
```


- **警告与提示表**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 严重 | `bytes(0b110)` 创建的是6个空字节，**不是**值0b110的bytes！正确写法：`bytes((0b110,))` 或 `b'\x06'` |
| ⚠️ 警告 | `int.from_bytes()` 必须手动指定 `byteorder` 和 `signed`，bytes 本身不记录这些信息 |
| ⚠️ 警告 | struct 解包时格式字符串必须与打包时**完全一致**，否则得到错误值或抛异常 |
| ⚠️ 警告 | `seek(whence=0)` 时偏移量必须非负，否则抛 `OSError`          |
| ⚠️ 警告 | bytes-like 对象不支持位运算，需转整数或迭代方案              |
| ⚠️ 警告 | 超过流结尾 seek 再写入数据会进入"黑洞"，不会被写入           |
| 💡 技巧 | 网络传输统一用大端序，本机处理用 `sys.byteorder`             |
| 💡 技巧 | 大二进制数据用 `memoryview` 切片，避免复制开销               |
| 💡 技巧 | 跨语言数据交换用 JSON，Python 内部用 pickle（需谨慎）        |
| 💡 技巧 | struct 的 `>ii?` 表示：大端序 + 2个int + 1个bool           |
| 💡 技巧 | 频繁使用同一格式字符串，用 `struct.Struct()` 预编译          |